<a href="https://colab.research.google.com/github/martinthuriaux/Auto-Interp-Causal-Validation/blob/main/04_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/martinthuriaux/CNN-Watch-vs-Do/blob/main/04_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — The watching test and the doing test (Stage 4 of PLAN.md, week 4)

Two grades per detector:

**Watching score** — does the label predict *firing*? Rank the held-out images by CLIP-similarity
to the label, rank them by the detector's actual firing, measure agreement (AUC). Scored on the
locked-away ImageNetV2 holdout half, so the label is graded on images it was NOT chosen from.

**Doing score** — does the label predict *damage*? Mean-ablate the detector, measure the per-class
accuracy drop over the evaluation set (the damage profile), then measure whether the damage lands
on classes semantically close to the label (CLIP text similarity). Also records raw importance
(total accuracy drop) separately.

**Gate A pilot first:** we run the doing test on one stage (256 units) before committing to all
1,920. If single-unit ablation does nothing for >90% of pilot units (redundancy), we pivot to
group ablation — decided here, before seeing the full results.

**Reused:** CLIP image embeddings (03), text embeddings (03), the activation matrix (03),
the frozen holdout split (01). **New caches:** watching scores, damage matrix, doing scores.

## 0. Bootstrap + guards

In [1]:
import os, json, hashlib, glob, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "no GPU — Runtime > Change runtime type > T4"
device = "cuda"

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/cnn_watch_vs_do")
RESULTS = PROJECT_ROOT / "results"
DATA = Path("/content/data"); DATA.mkdir(exist_ok=True)
(RESULTS / "scores").mkdir(parents=True, exist_ok=True)

CONFIG = json.load(open(RESULTS / "config.json"))
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]
STAGES = CONFIG["stages"]
STAGE_C = dict(zip(STAGES, CONFIG["expected_stage_channels"]))
index = json.load(open(RESULTS / "activations" / "image_index.json"))
N_IMAGES = len(index)

unit_order = [(s, c) for s in STAGES for c in range(STAGE_C[s])]
unit_col = {uc: j for j, uc in enumerate(unit_order)}
print("config hash:", CONFIG_HASH, "| units:", len(unit_order))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
config hash: 804f511675ac | units: 1920


## 1. Load the reused artifacts from week 3

The activation matrix (firing), CLIP image + text embeddings, the vocabulary, and both
label tables. All were cached in 03; nothing is recomputed.

In [2]:
A_max   = np.load(RESULTS / "embeddings" / "act_max_matrix.npy")     # [N_img, N_units] firing
img_emb = np.load(RESULTS / "embeddings" / "clip_img_emb.npy")       # [N_img, D] CLIP image
txt_emb = np.load(RESULTS / "embeddings" / "clip_txt_emb.npy")       # [V, D] CLIP word
vocab   = [w.strip() for w in open(RESULTS / "labels" / "vocab_20k.txt") if w.strip()]
dfA     = pd.read_json(RESULTS / "labels" / "labels_A.json")

# holdout mask: the ImageNetV2 half locked away in 01 (never used to pick labels)
holdout_ids = np.array([e["image_id"] for e in index if e.get("inv2_role") == "holdout"])
print(f"activation {A_max.shape}, img_emb {img_emb.shape}, txt_emb {txt_emb.shape}")
print(f"holdout images (for watching test): {len(holdout_ids)}")


activation (31178, 1920), img_emb (31178, 512), txt_emb (20000, 512)
holdout images (for watching test): 5000


## 2. Watching test — does the label predict firing on held-out images?

For each unit: embed its label, score each holdout image by similarity to the label, and ask
how well that similarity separates the unit's *top-firing* holdout images from the rest —
i.e. AUC of (label-similarity) as a classifier for (unit fired hard: top 1%). AUC 0.5 = label
predicts firing no better than chance; 1.0 = perfect. Vectorised over all units.

In [3]:
from sklearn.metrics import roc_auc_score

# label embedding per unit = CLIP text embedding of its label_A word
label_to_vocabidx = {w: i for i, w in enumerate(vocab)}
unitlabel_emb = np.stack([txt_emb[label_to_vocabidx[dfA.iloc[j].label_A]]
                          for j in range(len(dfA))])           # [N_units, D]

# similarity of each holdout image to each unit's label: [N_holdout, N_units]
sim = img_emb[holdout_ids] @ unitlabel_emb.T
act_hold = A_max[holdout_ids]                                   # [N_holdout, N_units] firing

TOPQ = CONFIG["top_firing_quantile"]     # top 1% = "unit fired hard"
watch_rows = []
for j, (s, c) in enumerate(unit_order):
    a = act_hold[:, j]
    thresh = np.quantile(a, 1 - TOPQ)
    y = (a >= thresh).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        auc = np.nan
    else:
        auc = roc_auc_score(y, sim[:, j])
    watch_rows.append({"stage": s, "channel": c, "watch_auc": auc})

dfW = pd.DataFrame(watch_rows)
dfW.to_json(RESULTS / "scores" / "watching_scores.json", orient="records", indent=1)
print("watching test done. median AUC:", round(dfW.watch_auc.median(), 3))
dfW.watch_auc.describe()


watching test done. median AUC: 0.554


,watch_auc
count,1920.000000
mean,0.565300
std,0.108404
min,0.247139
25%,0.490156
50%,0.554198
75%,0.629040
max,0.947523


## 3. Watching-test null (shuffled labels)

Sanity: if we pair each unit with a *random other unit's* label, the AUC should collapse to
~0.5. This calibrates what "the label genuinely predicts firing" means (H1).

In [4]:
rng = np.random.default_rng(0)
perm = rng.permutation(len(unit_order))
sim_null = img_emb[holdout_ids] @ unitlabel_emb[perm].T
null_aucs = []
for j in range(len(unit_order)):
    a = act_hold[:, j]; thresh = np.quantile(a, 1 - TOPQ)
    y = (a >= thresh).astype(int)
    if 0 < y.sum() < len(y):
        null_aucs.append(roc_auc_score(y, sim_null[:, j]))
print(f"real median AUC: {dfW.watch_auc.median():.3f}   "
      f"shuffled-label median AUC: {np.nanmedian(null_aucs):.3f}")
frac_beat = (dfW.watch_auc > np.nanquantile(null_aucs, 0.95)).mean()
print(f"units beating the 95th-pct null: {frac_beat:.1%}  (H1 sanity: want most units above)")


real median AUC: 0.554   shuffled-label median AUC: 0.474
units beating the 95th-pct null: 27.7%  (H1 sanity: want most units above)


## 4. Machinery for the doing test — model + ablation

Rebuild ResNet-18, load the evaluation images (the ImageNetV2 set, labelled in the model's
own 1,000 classes), and define mean-ablation: replace a unit's channel output with its mean
over the eval set (NOT zero — BatchNorm makes zeroing off-distribution).

In [5]:
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image

weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
model = resnet18(weights=weights).to(device).eval()

# evaluation set = ImageNetV2 images with their class labels (folder name = class index)
eval_items = [(e["image_id"], e["label"]) for e in index if e["dataset"] == "imagenetv2"]
print("eval images:", len(eval_items))

def load_raw(image_id):
    e = index[image_id]
    if e["dataset"] == "imagenetv2":
        return Image.open(DATA / e["ref"]).convert("RGB")
    split, j = e["ref"].split(":")
    return tv_sets[(e["dataset"], split)][int(j)][0].convert("RGB")

# NOTE: only imagenetv2 is needed for eval; ensure it's on local disk
INV2_DIR = DATA / "imagenetv2-matched-frequency-format-val"
if not INV2_DIR.exists():
    import shutil, tarfile, urllib.request
    tar_cache = PROJECT_ROOT / "cache" / "imagenetv2-matched-frequency.tar.gz"
    local_tar = DATA / "inv2.tar.gz"
    if tar_cache.exists(): shutil.copy(tar_cache, local_tar)
    else:
        url = ("https://huggingface.co/datasets/vaishaal/ImageNetV2/"
               "resolve/main/imagenetv2-matched-frequency.tar.gz")
        urllib.request.urlretrieve(url, local_tar); shutil.copy(local_tar, tar_cache)
    with tarfile.open(local_tar) as tf: tf.extractall(DATA)
    local_tar.unlink()

class EvalSet(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        img_id, lbl = self.items[i]
        return preprocess(load_raw(img_id)), lbl


eval images: 10000


In [6]:
# --- ablation via forward hooks: silence one channel of one stage by setting it to its mean ---
# First, measure each channel's mean activation over the eval set (needed for mean-ablation).
STAGE_MODULES = {s: dict(layer=s.split('.')[0], blk=int(s.split('.')[1])) for s in STAGES}
def get_block(stage):
    m = STAGE_MODULES[stage]
    return getattr(model, m["layer"])[m["blk"]]

# channel means: reuse A_max? No — A_max is spatial-max; ablation needs the spatial-mean value
# substituted at every spatial location. We approximate mean-ablation by replacing the whole
# channel with its per-channel mean activation, computed here over a sample of eval images.
@torch.no_grad()
def compute_channel_means(sample_n=1000):
    means = {s: torch.zeros(STAGE_C[s], device=device) for s in STAGES}
    cnt = 0
    capt = {}
    hooks = [get_block(s).register_forward_hook(
        (lambda st: (lambda mod, i, o: capt.__setitem__(st, o)))(s)) for s in STAGES]
    loader = DataLoader(EvalSet(eval_items[:sample_n]), batch_size=128, num_workers=2)
    for x, _ in loader:
        model(x.to(device))
        for s in STAGES:
            means[s] += capt[s].mean(dim=(0, 2, 3)) * x.size(0)
        cnt += x.size(0)
    for h in hooks: h.remove()
    for s in STAGES: means[s] /= cnt
    return means

channel_means = compute_channel_means()
print("channel means computed for mean-ablation")


channel means computed for mean-ablation


In [7]:
import numpy as np, torch

@torch.no_grad()
def eval_accuracy_per_class(ablate=None, subset=None, batch_size=256):
    items = subset if subset is not None else eval_items
    labels_all = torch.tensor([lbl for _, lbl in items])
    correct = torch.zeros(1000); total = torch.zeros(1000)
    total.index_add_(0, labels_all, torch.ones(len(items)))   # class counts, vectorised

    handle = None
    if ablate is not None:
        s, c = ablate
        mval = channel_means[s][c]
        def hook(mod, inp, out):
            out[:, c, :, :] = mval
            return out
        handle = get_block(s).register_forward_hook(hook)

    loader = DataLoader(EvalSet(items), batch_size=batch_size, num_workers=2)
    ptr = 0
    for x, y in loader:
        pred = model(x.to(device)).argmax(1).cpu()
        hits = (pred == y).float()                            # [B] correct-or-not
        correct.index_add_(0, y, hits)                        # vectorised tally, no Python loop
        ptr += len(y)
    if handle: handle.remove()
    return correct.numpy(), total.numpy()

## 5. GATE A pilot — doing test on ONE stage (layer3.0, 256 units)

Before committing to all 1,920 ablations, run the doing test on one mid-stage and check the
redundancy risk. If >90% of these units show total damage indistinguishable from the
random-ablation null, single-unit ablation is uninformative and we PIVOT to group ablation
(pre-committed). Damage measured on a 2,000-image stratified subset for speed.

In [8]:
# ---- stratified 1k screening subset, preprocessed ONCE into a GPU tensor ----
import torch, time
rng = np.random.default_rng(CONFIG["seed"])
by_class = {}
for it in eval_items:
    by_class.setdefault(it[1], []).append(it)
screen_items = []
for lbl, its in by_class.items():
    screen_items += list(rng.permutation(its)[:1])
print("screening subset:", len(screen_items), "images — preprocessing once...")

# decode + preprocess all screening images a single time, cache on GPU
_xs, _ys = [], []
for img_id, lbl in screen_items:
    _xs.append(preprocess(load_raw(img_id)))
    _ys.append(lbl)
screen_x = torch.stack(_xs).to(device)          # [N, 3, 224, 224] on GPU, decoded once
screen_y = torch.tensor(_ys)
print("cached screening tensor:", tuple(screen_x.shape))

@torch.no_grad()
def total_damage_fast(ablate=None, bs=256):
    handle = None
    if ablate is not None:
        s, c = ablate
        mval = channel_means[s][c]
        def hook(mod, inp, out):
            out[:, c, :, :] = mval; return out
        handle = get_block(s).register_forward_hook(hook)
    correct = 0
    for i in range(0, len(screen_x), bs):
        pred = model(screen_x[i:i+bs]).argmax(1).cpu()
        correct += (pred == screen_y[i:i+bs]).sum().item()
    if handle: handle.remove()
    return correct / len(screen_y)

sc_base_acc = total_damage_fast(None)
def total_damage(ablate): return sc_base_acc - total_damage_fast(ablate)
print(f"screening baseline acc: {sc_base_acc:.3f}")

# ---- Gate A pilot ----
PILOT_STAGE = "layer3.0"
pilot_units = [(PILOT_STAGE, c) for c in range(STAGE_C[PILOT_STAGE])]

t0 = time.time()
pilot_damage = np.array([total_damage(u) for u in pilot_units])
print(f"pilot done: {len(pilot_units)} units in {time.time()-t0:.0f}s")

# random-ablation null: pick stage ONCE, then a valid channel for THAT stage
null_damage = []
for _ in range(20):
    s = rng.choice(STAGES)                       # one pick
    c = int(rng.integers(STAGE_C[s]))            # channel valid for that stage
    null_damage.append(total_damage((s, c)))
null_damage = np.array(null_damage)
null_95 = np.quantile(null_damage, 0.95)

frac_inert = (pilot_damage <= null_95).mean()
print(f"\npilot median damage: {np.median(pilot_damage):+.4f}")
print(f"random-null 95th pct: {null_95:+.4f}")
print(f"fraction inert (at/below null): {frac_inert:.1%}")
print("\n>>> GATE A:", "PIVOT to group ablation" if frac_inert > 0.90
      else "PROCEED with single-unit ablation")

screening subset: 1000 images — preprocessing once...
cached screening tensor: (1000, 3, 224, 224)
screening baseline acc: 0.579
pilot done: 256 units in 234s

pilot median damage: +0.0010
random-null 95th pct: +0.0082
fraction inert (at/below null): 99.6%

>>> GATE A: PIVOT to group ablation


## 6. PIVOT: group ablation (Gate A fired — single units are redundant)

Gate A found ~99% of single units causally inert: the network spreads each concept across many
redundant units, so silencing one changes nothing. We therefore ablate **groups** of units that
share a concept, and measure the group's damage. Two grouping methods, compared:

- **Method 1 — identical label word:** all units Labeler A gave the same word (all "dog" units).
- **Method 2 — CLIP-similarity clusters:** merge near-synonym labels ("dog"/"puppy"/"retriever")
  into one concept via clustering of the label-word embeddings.

If Method 2 (merged concepts) shows stronger damage than Method 1 (fragmented words), that is
direct evidence concepts live across synonym-labeled units.

Groups are defined from Labeler A (all 1,920 units). We reuse the fast GPU-cached screening
tensor from cell 5, so each group ablation is milliseconds.

In [9]:
# fast multi-channel ablation on the cached screening tensor (reuses screen_x/screen_y from cell 5)
@torch.no_grad()
def total_damage_group(units, bs=256):
    """units: list of (stage, channel). Silence ALL of them at once, return accuracy drop."""
    by_stage = {}
    for s, c in units:
        by_stage.setdefault(s, []).append(c)
    handles = []
    for s, chans in by_stage.items():
        chans_t = torch.tensor(chans, device=device)
        means_t = channel_means[s][chans_t]            # [k]
        def make_hook(ct, mt):
            def hook(mod, inp, out):
                out[:, ct, :, :] = mt[None, :, None, None]
                return out
            return hook
        handles.append(get_block(s).register_forward_hook(make_hook(chans_t, means_t)))
    correct = 0
    for i in range(0, len(screen_x), bs):
        pred = model(screen_x[i:i+bs]).argmax(1).cpu()
        correct += (pred == screen_y[i:i+bs]).sum().item()
    for h in handles: h.remove()
    return sc_base_acc - correct / len(screen_y)

# per-class version (needed for the doing score) — damage vector over 1000 classes
@torch.no_grad()
def per_class_damage_group(units, bs=256):
    by_stage = {}
    for s, c in units:
        by_stage.setdefault(s, []).append(c)
    handles = []
    for s, chans in by_stage.items():
        chans_t = torch.tensor(chans, device=device)
        means_t = channel_means[s][chans_t]
        def make_hook(ct, mt):
            def hook(mod, inp, out):
                out[:, ct, :, :] = mt[None, :, None, None]; return out
            return hook
        handles.append(get_block(s).register_forward_hook(make_hook(chans_t, means_t)))
    correct = torch.zeros(1000); total = torch.zeros(1000)
    total.index_add_(0, screen_y, torch.ones(len(screen_y)))
    for i in range(0, len(screen_x), bs):
        pred = model(screen_x[i:i+bs]).argmax(1).cpu()
        hits = (pred == screen_y[i:i+bs]).float()
        correct.index_add_(0, screen_y[i:i+bs], hits)
    for h in handles: h.remove()
    acc = np.divide(correct.numpy(), total.numpy(), out=np.zeros(1000), where=total.numpy()>0)
    return sc_base_acc_perclass - acc            # positive = accuracy lost, per class

# baseline per-class accuracy on the screening set (once)
_c = torch.zeros(1000); _t = torch.zeros(1000)
_t.index_add_(0, screen_y, torch.ones(len(screen_y)))
for i in range(0, len(screen_x), 256):
    pred = model(screen_x[i:i+256]).argmax(1).cpu()
    _c.index_add_(0, screen_y[i:i+256], (pred == screen_y[i:i+256]).float())
sc_base_acc_perclass = np.divide(_c.numpy(), _t.numpy(), out=np.zeros(1000), where=_t.numpy()>0)
print("group-ablation machinery ready")


group-ablation machinery ready


### 6a. Method 1 — groups by identical label word

In [10]:
from collections import defaultdict

# map each label word -> list of units with that word (Labeler A)
word_groups = defaultdict(list)
for _, r in dfA.iterrows():
    word_groups[r.label_A].append((r.stage, r.channel))

# only test groups with >= 2 units (a group of 1 is just single-unit, already shown inert)
word_groups = {w: us for w, us in word_groups.items() if len(us) >= 2}
print(f"{len(word_groups)} label-word groups with >=2 units "
      f"(covering {sum(len(u) for u in word_groups.values())} units)")

import time
t0 = time.time()
m1_rows = []
for w, units in sorted(word_groups.items(), key=lambda kv: -len(kv[1])):
    dmg = total_damage_group(units)
    m1_rows.append({"group": w, "n_units": len(units), "total_damage": dmg})
dfM1 = pd.DataFrame(m1_rows).sort_values("total_damage", ascending=False)
print(f"Method 1 done: {len(dfM1)} groups in {time.time()-t0:.0f}s")

# how many groups now beat the single-unit null? (shows grouping recovers causal signal)
frac_alive = (dfM1.total_damage > null_95).mean()
print(f"groups with damage above the single-unit null: {frac_alive:.1%} "
      f"(vs ~0.4% of single units)")
dfM1.head(15)

332 label-word groups with >=2 units (covering 1496 units)
Method 1 done: 332 groups in 322s
groups with damage above the single-unit null: 16.6% (vs ~0.4% of single units)


,group,n_units,total_damage
5,patterns,28,0.155
20,hypnosis,12,0.142
0,dots,41,0.086
31,particles,9,0.064
16,mesh,14,0.061
54,turbulence,6,0.056
53,stains,6,0.054
21,wrought,11,0.051
1,puppy,33,0.046
14,triangle,15,0.039


### 6b. Method 2 — groups by CLIP-similarity clusters of labels

Cluster the distinct label words by their CLIP text-embedding similarity, so near-synonyms merge
into one concept. Then ablate each cluster's units together.

In [11]:
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict
import time

distinct_words = sorted(set(dfA.label_A))
word_emb = np.stack([txt_emb[label_to_vocabidx[w]] for w in distinct_words])

# merge words whose CLIP cosine similarity exceeds a threshold
# metric="cosine" clusters on cosine DISTANCE = 1 - similarity
SIM_THRESHOLD = 0.92                      # only near-identical words merge
clust = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1 - SIM_THRESHOLD,
    metric="cosine",
    linkage="average")
word_cluster = clust.fit_predict(word_emb)
word2clust = {w: int(k) for w, k in zip(distinct_words, word_cluster)}
print(f"SIM_THRESHOLD={SIM_THRESHOLD} -> {clust.n_clusters_} clusters "
      f"from {len(distinct_words)} distinct words")

clust_groups = defaultdict(list); clust_words = defaultdict(list)
for _, r in dfA.iterrows():
    k = word2clust[r.label_A]
    clust_groups[k].append((r.stage, r.channel))
    clust_words[k].append(r.label_A)
clust_groups = {k: us for k, us in clust_groups.items() if len(us) >= 2}

# diagnostic: are the biggest clusters coherent, or over-merged?
print("\nLargest clusters (check coherence):")
for k, units in sorted(clust_groups.items(), key=lambda kv: -len(kv[1]))[:8]:
    words = pd.Series(clust_words[k]).value_counts().head(6).index.tolist()
    print(f"  cluster {k}: {len(units)} units | {', '.join(words)}")

t0 = time.time()
m2_rows = []
for k, units in clust_groups.items():
    dmg = total_damage_group(units)
    top_words = pd.Series(clust_words[k]).value_counts().head(4).index.tolist()
    m2_rows.append({"cluster": k, "n_units": len(units),
                    "example_words": ", ".join(top_words), "total_damage": dmg})
dfM2 = pd.DataFrame(m2_rows).sort_values("total_damage", ascending=False)
print(f"\nMethod 2 done: {len(dfM2)} clusters in {time.time()-t0:.0f}s")
dfM2.head(15)

SIM_THRESHOLD=0.92 -> 709 clusters from 756 distinct words

Largest clusters (check coherence):
  cluster 472: 41 units | dots
  cluster 7: 40 units | puppy, puppies
  cluster 156: 33 units | patterns, pattern
  cluster 330: 33 units | terrier
  cluster 14: 31 units | huskies, husky
  cluster 267: 30 units | retriever
  cluster 280: 29 units | newfoundland
  cluster 23: 25 units | flowers, flower, floral

Method 2 done: 334 clusters in 325s


,cluster,n_units,example_words,total_damage
2,156,33,"patterns, pattern",0.191
8,210,12,hypnosis,0.142
6,472,41,dots,0.086
11,237,9,particles,0.064
5,136,14,mesh,0.061
21,83,6,turbulence,0.056
9,158,6,stains,0.054
28,45,11,wrought,0.051
39,106,15,triangle,0.039
93,14,31,"huskies, husky",0.038


### 6c. Method 1 vs Method 2 — does merging synonyms strengthen the causal signal?

In [12]:
print("Median group damage:")
print(f"  Method 1 (identical word):   {dfM1.total_damage.median():+.4f}")
print(f"  Method 2 (concept cluster):  {dfM2.total_damage.median():+.4f}")
print(f"\nMax group damage:")
print(f"  Method 1: {dfM1.total_damage.max():+.4f}  ({dfM1.iloc[0].group})")
print(f"  Method 2: {dfM2.total_damage.max():+.4f}  ({dfM2.iloc[0].example_words})")
print(f"\nSingle-unit null 95th pct (reference): {null_95:+.4f}")
print("\nIf Method 2 medians/maxima exceed Method 1, merging synonym-labelled units "
      "recovers concept-level causal structure the vocabulary had fragmented.")


Median group damage:
  Method 1 (identical word):   +0.0030
  Method 2 (concept cluster):  +0.0030

Max group damage:
  Method 1: +0.1550  (patterns)
  Method 2: +0.1910  (patterns, pattern)

Single-unit null 95th pct (reference): +0.0082

If Method 2 medians/maxima exceed Method 1, merging synonym-labelled units recovers concept-level causal structure the vocabulary had fragmented.



## 7. Group-level doing score — does a concept's damage land on its classes?

For each concept group (Method 1 words), get its per-class damage vector, and measure whether
the damage concentrates on ImageNet classes semantically close to the group's label. This is the
group-level version of the doing score: "when we silence all the 'dog' units, does *dog*
classification specifically collapse?" High alignment = the label describes what the network
uses that group of units *for*.

In [ ]:
# ============================================================
#  CELL 7 (clean rewrite): doing score, fast + crash-proof
# ============================================================
import numpy as np, torch, time, json, gc
from scipy.stats import spearmanr

# free memory from the watching test
for _v in ["A_max", "img_emb", "sim", "act_hold"]:
    if _v in dir(): del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

# ---------- 1. decode 10 imgs/class ONCE to a memmap on Drive ----------
MMAP  = RESULTS / "scores" / "dense_images.dat"
LBLS  = RESULTS / "scores" / "dense_labels.npy"

rng = np.random.default_rng(CONFIG["seed"])
by_class = {}
for it in eval_items:
    by_class.setdefault(it[1], []).append(it)
dense_items = []
for lbl, its in by_class.items():
    dense_items += list(rng.permutation(its)[:10])
N = len(dense_items)
dense_y = torch.tensor([l for _, l in dense_items])
np.save(LBLS, dense_y.numpy())

if not MMAP.exists():
    print(f"decoding {N} images once -> memmap (~3-5 min, one time only)...")
    mm = np.memmap(MMAP, dtype=np.float16, mode="w+", shape=(N, 3, 224, 224))
    t0 = time.time()
    for i, (img_id, _) in enumerate(dense_items):
        mm[i] = preprocess(load_raw(img_id)).half().numpy()
        if (i+1) % 2000 == 0:
            mm.flush(); print(f"  decoded {i+1}/{N}, {time.time()-t0:.0f}s")
    mm.flush(); del mm
    print("memmap written to Drive")
else:
    print("memmap already exists — skipping decode")
dense_mm = np.memmap(MMAP, dtype=np.float16, mode="r", shape=(N, 3, 224, 224))

# ---------- 2. fast per-class accuracy (reads pre-decoded arrays) ----------
@torch.no_grad()
def per_class_acc(ablate_units=None, bs=512):
    handles = []
    if ablate_units:
        by_stage = {}
        for s, c in ablate_units:
            by_stage.setdefault(s, []).append(c)
        for s, chans in by_stage.items():
            ct = torch.tensor(chans, device=device); mt = channel_means[s][ct]
            def mk(ct, mt):
                def hook(mod, inp, out):
                    out[:, ct, :, :] = mt[None, :, None, None]; return out
                return hook
            handles.append(get_block(s).register_forward_hook(mk(ct, mt)))
    correct = torch.zeros(1000); total = torch.zeros(1000)
    for i in range(0, N, bs):
        x = torch.from_numpy(np.asarray(dense_mm[i:i+bs])).float().to(device)
        y = dense_y[i:i+bs]
        pred = model(x).argmax(1).cpu()
        correct.index_add_(0, y, (pred == y).float())
        total.index_add_(0, y, torch.ones(len(y)))
    for h in handles: h.remove()
    return np.divide(correct.numpy(), total.numpy(), out=np.zeros(1000), where=total.numpy()>0)

base_acc = per_class_acc(None)
def damage(units): return base_acc - per_class_acc(units)
print(f"dense baseline acc: {base_acc.mean():.3f}")

# ---------- 3. POSITIVE CONTROL (checkpointed scan) ----------
CKPT = RESULTS / "scores" / "ctrl_scan.json"
good = [i for i in range(1000) if base_acc[i] >= 0.8]
CTRL = good[len(good)//2]
print(f"control class: '{classnames[CTRL]}' (base acc {base_acc[CTRL]:.2f})")

scan_units = [(s, c) for s in ["layer4.0", "layer4.1"] for c in range(STAGE_C[s])]
scores = json.load(open(CKPT)) if CKPT.exists() else {}
t0 = time.time()
for k, (s, c) in enumerate(scan_units):
    key = f"{s}_{c}"
    if key in scores: continue
    scores[key] = float(damage([(s, c)])[CTRL])
    if (k+1) % 50 == 0:
        json.dump(scores, open(CKPT, "w")); print(f"  scan {k+1}/{len(scan_units)}, {time.time()-t0:.0f}s")
json.dump(scores, open(CKPT, "w"))

ranked = sorted(scores.items(), key=lambda x: -x[1])
ctrl_group = [tuple([k.rsplit("_",1)[0], int(k.rsplit("_",1)[1])]) for k, _ in ranked[:20]]
cdmg = damage(ctrl_group)
ctrl_rho = spearmanr(cdmg, class_emb @ class_emb[CTRL]).statistic
print(f"\n>>> POSITIVE CONTROL doing_rho = {ctrl_rho:+.3f}")
print(f"    worst-hit = '{classnames[int(np.argmax(cdmg))]}' (want '{classnames[CTRL]}')")

# ---------- 4. real group doing scores ----------
rows = []
for w, units in word_groups.items():
    d = damage(units)
    rho = np.nan if d.std() < 1e-9 else spearmanr(d, class_emb @ txt_emb[label_to_vocabidx[w]]).statistic
    rows.append({"group": w, "n_units": len(units), "total_damage": float(d.sum()),
                 "doing_rho": rho, "worst_hit_class": classnames[int(np.argmax(d))]})
dfDoing = pd.DataFrame(rows).sort_values("doing_rho", ascending=False)
dfDoing.to_json(RESULTS / "scores" / "group_doing_scores_dense.json", orient="records", indent=1)
print(f"\nmedian doing_rho: {dfDoing.doing_rho.median():+.3f}")
print("\nStrongest passes:"); display(dfDoing.head(12))
print("\nWatched-but-not-doing:"); display(dfDoing.dropna(subset=["doing_rho"]).tail(12))

decoding 10000 images once -> memmap (~3-5 min, one time only)...
  decoded 2000/10000, 31s
  decoded 4000/10000, 49s
  decoded 6000/10000, 68s
  decoded 8000/10000, 87s
  decoded 10000/10000, 109s
memmap written to Drive


/tmp/ipykernel_30689/1369207102.py:58: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.asarray(dense_mm[i:i+bs])).float().to(device)


dense baseline acc: 0.572
control class: 'dromedary' (base acc 0.90)
  scan 50/1024, 700s
  scan 100/1024, 1379s
  scan 150/1024, 2063s
  scan 200/1024, 2747s
  scan 250/1024, 3435s
  scan 300/1024, 4135s
